In [1]:
# Clone your repository
!git clone https://github.com/shadyOg/fyp-dtcr-unet.git
%cd fyp-dtcr-unet/dtcr-unet
!pip install -q -r requirements.txt
print("Repository cloned and dependencies installed successfully!")


Cloning into 'fyp-dtcr-unet'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 104 (delta 31), reused 92 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 44.52 KiB | 5.57 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/kaggle/working/fyp-dtcr-unet/dtcr-unet
Repository cloned and dependencies installed successfully!


In [2]:
import os

print("Available input datasets in /kaggle/input:")
print(os.listdir('/kaggle/input'))

# Path where your preprocessed slices will be stored
PROCESSED_DIR = '/kaggle/working/data/processed'

# Run preprocessing on MosMedData
!python data/preprocessing.py \
    --mosmed /kaggle/input/mosmeddata-chest-ct-scans-with-covid19 \
    --out {PROCESSED_DIR} \
    --size 256 \
    --context 1 \
    --labeled-ratio 0.4


Available input datasets in /kaggle/input:
['datasets']
Found 0 paired MosMed volumes.
Traceback (most recent call last):
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/data/preprocessing.py", line 324, in <module>
    main()
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/data/preprocessing.py", line 313, in main
    process_mosmed_dataset(
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/data/preprocessing.py", line 189, in process_mosmed_dataset
    raise FileNotFoundError(f"No paired MosMed studies/masks found in {mosmed_root}")
FileNotFoundError: No paired MosMed studies/masks found in /kaggle/input/mosmeddata-chest-ct-scans-with-covid19


In [3]:
%cd /kaggle/working/fyp-dtcr-unet/dtcr-unet
!git pull origin main


/kaggle/working/fyp-dtcr-unet/dtcr-unet
From https://github.com/shadyOg/fyp-dtcr-unet
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
PROCESSED_DIR = '/kaggle/working/data/processed'

# Simply pass /kaggle/input -- it will now automatically discover all files recursively!
!python data/preprocessing.py \
    --mosmed /kaggle/input \
    --out {PROCESSED_DIR} \
    --size 256 \
    --context 1 \
    --labeled-ratio 0.4


Found 50 paired MosMed volumes.
Total annotated studies with lesions: 50
Split breakdown: Train=30 studies, Val=10 studies, Test=10 studies.
Processing test split: 100%|████████████████████| 10/10 [00:04<00:00,  2.29it/s]

Preprocessing complete!
{
  "dataset": "MosMedData",
  "target_size": [
    256,
    256
  ],
  "total_extracted_slices": 1007,
  "splits": {
    "train": 640,
    "val": 181,
    "test": 186
  },
  "semi_supervised_breakdown": {
    "train_labeled": 227,
    "train_unlabeled": 413
  },
  "labeled_ratio": 0.4,
  "seed": 42
}


In [5]:
%cd /kaggle/working/fyp-dtcr-unet/dtcr-unet

!python train.py \
    --data /kaggle/working/data/processed \
    --epochs 80 \
    --batch-size 16 \
    --lr 0.001 \
    --labeled-ratio 0.4 \
    --checkpoints /kaggle/working/checkpoints \
    --outputs /kaggle/working/outputs


/kaggle/working/fyp-dtcr-unet/dtcr-unet
Using device: cuda
Data ready: 640 train samples, 181 val samples.

Starting DTCR-U-Net training for 80 epochs...

Epoch 1/80 [Train]:   0%|                                | 0/40 [00:02<?, ?it/s]
Traceback (most recent call last):
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 246, in <module>
    main()
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 242, in main
    train(cfg)
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 163, in train
    train_losses = train_one_epoch(
                   ^^^^^^^^^^^^^^^^
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/train.py", line 50, in train_one_epoch
    f1_logits, f2_lsf, f2_trans = model(images)
                                  ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/l

In [6]:
!python evaluate.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pth \
    --data /kaggle/working/data/processed \
    --out /kaggle/working/outputs/evaluation \
    --batch-size 16


Loading checkpoint from: /kaggle/working/checkpoints/best_model.pth
Traceback (most recent call last):
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/evaluate.py", line 130, in <module>
    main()
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/evaluate.py", line 121, in main
    evaluate_checkpoint(
  File "/kaggle/working/fyp-dtcr-unet/dtcr-unet/evaluate.py", line 36, in evaluate_checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=dev)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__in